<a href="https://colab.research.google.com/github/juanjodoblasm/proyecto_algoritmos_optimizacion/blob/main/SEMINARIO/trabajo_practico-problema1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Algoritmos de optimización - Proyecto de programación
Nombre y Apellidos: Christian Dario Barahona Pesantes y Juan José Doblas Martínez

[GitHub](https://github.com/juanjodoblasm/proyecto_algoritmos_optimizacion)

Problema:
1. Sesiones de doblaje

Descripción del problema:

Se precisa coordinar el doblaje de una película. Los actores del doblaje deben coincidir en las tomas en las que sus personajes aparecen juntos en las diferentes tomas. Los actores de doblaje cobran todos la misma cantidad por cada día que deben desplazarse hasta el estudio de grabación independientemente del número de tomas que se graben. No es posible grabar más de 6 tomas por día. El objetivo es planificar las sesiones por día de manera que el gasto por los servicios de los actores de doblaje sea el menor posible. Los datos son:
- Número de actores: 10
- Número de tomas: 30
- Actores/Tomas: https://bit.ly/36D8IuK
  - 1 indica que el actor participa en la toma,
  - 0 en caso contrario.

Librerías que usaremos durante el trabajo:

In [38]:
import math
from functools import lru_cache
import pandas as pd
import numpy as np
import copy
from itertools import combinations

(*)¿Cuántas posibilidades hay sin tener en cuenta las restricciones?

Respuesta

Para saber cuántas posibilidades existen sin tener en cuenta las restricciones, debemos contar el número de particiones de un conjunto de 30 elementos (el conjunto de tomas), ya que queremos saber en cuántas formas podemos dividir las 30 tomas en hasta 30 días. Esto es porque suponemos que cada día se rueda al menos una toma hasta finalizar el rodaje. Además, también consideramos que si seleccionamos dos o más tomas en un día, no nos importa el orden en el que se rueden y tampoco nos importa el día que se asigne a cada grupo de tomas. Al final, lo relevante es la función de coste, que es el número de actores que llevamos al set de rodaje cada día y esto no cambia con el orden de las tomas de un mismo día o con el día que asignemos a cada grupo de tomas.

Este número se conoce como el trigésimo número de Bell, $B_{30}$. Los números de Bell siguen la siguiente fórmula recursiva:
$$
\begin{align*}
    B_0 &= 1, \\
    B_n &= \sum_{k=0}^{n-1}\binom{n-1}{k}B_k, \forall n \ge 1.
\end{align*}
$$
Esta fórmula puede explicarse observando que, a partir de una partición arbitraria de $n$ elementos, la eliminación del conjunto que contiene un elemento fijado deja una partición de un conjunto menor de $k$ elementos para algún número $k \in \{0, 1, \ldots, n-1\}$. Hay $\binom{n-1}{k}$ opciones para los $k$ elementos que quedan después de que se elimine un conjunto, y $B_k$ opciones de cómo dividirlos.

Así, podemos calcular el número de posibilidades con el siguiente código.

In [39]:
# Recurrencia de los números de Bell
@lru_cache(maxsize=31)
def numero_bell(n):
    if n == 0:
        return 1
    else:
        num = 0
        for k in range(n):
            num += math.comb(n-1, k) * numero_bell(k)
        return num

numero_bell(30)

846749014511809332450147

Esto es, aproximadamente 847 mil trillones de posibilidades, es decir, $847\cdot10^{21}$.

¿Cuántas posibilidades hay teniendo en cuenta todas las restricciones?

Respuesta

La única restricción que tenemos nos dice que ningún subconjunto de la partición del conjunto de tomas puede superar los 6 elementos.

De forma general, podemos definir una modificación del número de Bell que cuente el número de particiones posibles de un conjunto de $n$ elementos con, como máximo, $m$ elementos por subconjunto. Esto también se puede definir de manera recursiva:
$$
\begin{align*}
    B_0 &= 1, \\
    B_n^{(m)} &= \sum_{k=\max(n-m, 0)}^{n-1}\binom{n-1}{k}B_k^{(m)}, \forall n \ge 1.
\end{align*}
$$
En este caso, si $n \le m$, la fórmula es la misma que la del número de Bell, ya que la restricción se cumple trivialmente. Por otro lado, cuando $n > m$, observamos que el sumatorio solamente recorre $m$ términos. Esto se debe a que, a partir de una partición arbitraria de $n$ elementos, la eliminación del conjunto que contiene un elemento fijado deja una partición restringida de un conjunto menor de $k$ elementos para algún número $k$, a priori entre $0$ y $n-1$. Sin embargo, el conjunto eliminado, que tiene $n-k$ elementos, tampoco puede tener más de $m$ elementos, por lo que se cumple la desigualdad $n-k \le m$, de donde $k \ge n-m$; junto con lo anterior, obtenemos $n-m \le k \le n-1$.

De esta forma, hay $\binom{n-1}{k}$ opciones para los $k$ elementos que quedan después de que se elimine un conjunto, y $B_k^{(m)}$ opciones de cómo dividirlos.

Así, podemos calcular el número de posibilidades con el siguiente código.

In [ ]:
# Recurrencia de los números de Bell modificados
@lru_cache(maxsize=31)
def numero_bell_modificado(n, m):
    if n == 0:
        return 1
    else:
        num = 0
        for k in range(max(n-m, 0), n):
            num += math.comb(n-1, k) * numero_bell_modificado(k, m)
        return num

numero_bell_modificado(30, 6)

726391948970868949621309

Esto es, aproximadamente 726 mil trillones de posibilidades, es decir, $726\cdot10^{21}$.

Modelo para el espacio de soluciones

(*) ¿Cual es la estructura de datos que mejor se adapta al problema? Arguméntalo.(Es posible que hayas elegido una al principio y veas la necesidad de cambiar, arguméntalo)


Respuesta

El objetivo del problema es planificar las sesiones por día, es decir, decidir una partición del conjunto de 30 tomas con la restricción de que no puede haber más de 6 tomas por día (subconjunto). Esto significa que no nos importa el orden de las tomas que se rueden en un mismo día, ni tampoco el día en que se ruede cada grupo de tomas, ya que la función de coste no cambia con esas variaciones. Por eso, la estructura de datos que mejor se adapta a organizar las tomas es el `set`. Sin embargo, como Python no permite crear `sets` de `sets`, la planificación la guardaremos en una `list`.

Por otro lado, para tratar con los actores, también usaremos `sets`, porque dada una agrupación de tomas, para saber qué actores debemos llevar al estudio de grabación ese día, hay que hacer la unión de los conjuntos de actores que participan en cada una de esas tomas.

Según el modelo para el espacio de soluciones:

(*)¿Cual es la función objetivo?

Respuesta

La función objetivo nos debe dar el gasto por los servicios de los actores de doblaje. Como todos los actores cobran la misma cantidad, podemos suponer que cada actor tiene un coste $1$.

Como el número de actores que debemos llevar al estudio de grabación depende únicamente de las tomas que hayamos decidido grabar ese día y a cada actor le asignamos coste $1$, nuestra función objetivo tomará como variable la partición de las tomas y nos devolverá el coste asociado a esa partición sumando el número de actores que se requieren en cada conjunto de tomas, que se calcula haciendo la unión de los actores que participan en cada toma de ese conjunto.

Formalmente, sean $T = \{t_i | i = 1, \ldots, n\}$ el conjunto de tomas y $A = \{a_j | j = 1, \ldots, p\}$ el conjunto de actores. Definimos $A_i \subseteq A$ como el subconjunto de actores que participan en la toma $t_i$. También definimos $\mathbb{P}^{(m)}(T)$ como el conjunto de todas las particiones de $T$ con, como máximo, $m$ elementos por subconjunto (por lo que $\left|\mathbb{P}^{(m)}(T)\right| = B_n^{(m)}$). Nuestra función objetivo es
$$
f : \mathbb{P}^{(6)}(T) \to \mathbb{N},
$$
definida como
$$
f\left(\{G_1, \ldots, G_q\}\right) = \sum_{k = 1}^q\left|\bigcup_{t_i \in G_k}A_i\right|
$$

(*)¿Es un problema de maximización o minimización?

Respuesta

El objetivo del problema nos pide que el gasto por los servicios de los actores de doblaje sea el menor posible, por lo tanto, es un problema de minimización.

Formalmente, dado que lo que buscamos no es únicamente el valor mínimo del coste sino la partición concreta que lo alcanza, buscamos
$$
\underset{P \in \mathbb{P}^{(6)}(T)}{\arg\min}\ f(P)
$$

Diseña un algoritmo para resolver el problema por fuerza bruta

Respuesta

Primero crearemos dos funciones: una que nos devuelva todas las particiones de un conjunto (para recorrer por fuerza bruta) y nuestra función objetivo.

In [41]:
def generador_particiones(conjunto, m):
    """
    Genera de forma recursiva todas las particiones posibles de `subset`
    en las que ningún grupo (subconjunto) supera los `m` elementos.

    input:
        conjunto (set): conjunto de elementos a particionar.
        m (int): tamaño máximo permitido por grupo de la partición.
    output:
        Iterador de particiones; cada partición es una lista de conjuntos
        (list[set]), uno por grupo.
    """
    # Caso base: conjunto vacío
    if not conjunto:
        # Devolvemos cada partición como una lista de conjuntos porque set no admite
        # otros sets como elementos al no ser estos hashables
        yield []
        return

    # Fijamos un elemento y lo quitamos del conjunto
    elemento = next(iter(conjunto))
    resto = conjunto - {elemento}

    # Recorremos las posibles combinaciones de k elementos
    for i in range(min(m - 1, len(resto)) + 1):
        for combinacion in combinations(resto, i):
            grupo = {elemento} | set(combinacion)
            nuevo_resto = resto - set(combinacion)
            # Generamos las particiones de forma recursiva
            for particion_resto in generador_particiones(nuevo_resto, m):
                yield [grupo] + particion_resto

def f_objetivo(particion, tomas):
    """
    Calcula el coste de una partición: la suma, para cada grupo (día),
    del número de actores distintos que se necesitan ese día.

    input:
        particion (list[set]): partición de tomas; cada grupo es un día.
        tomas (dict): diccionario toma -> set de actores que participan.
    output:
        int: coste total de la partición (número de actor-días).
    """
    coste = 0

    # Para cada grupo de tomas calculamos los actores necesarios
    for grupo in particion:
        actores = set()

        for toma in grupo:
            actores |= tomas[toma]

        # Sumamos el coste de este grupo de tomas
        coste += len(actores)

    return coste

Veamos un ejemplo de todas las posibles particiones de un conjunto de 4 elementos con, como máximo, 2 elementos por subconjunto

In [42]:
i = 1
for particion in generador_particiones({1, 2, 3, 4}, 2):
    print(f"Partición {i}: {particion}")
    i += 1

print(f"Total de particiones: {numero_bell_modificado(4, 2)}")

Partición 1: [{1}, {2}, {3}, {4}]
Partición 2: [{1}, {2}, {3, 4}]
Partición 3: [{1}, {2, 3}, {4}]
Partición 4: [{1}, {2, 4}, {3}]
Partición 5: [{1, 2}, {3}, {4}]
Partición 6: [{1, 2}, {3, 4}]
Partición 7: [{1, 3}, {2}, {4}]
Partición 8: [{1, 3}, {2, 4}]
Partición 9: [{1, 4}, {2}, {3}]
Partición 10: [{1, 4}, {2, 3}]
Total de particiones: 10


Ahora cargamos y limpiamos los datos con los que tenemos que trabajar

In [43]:
# 1. Cargar datos del CSV (fila 1 como cabecera)
url = 'https://raw.githubusercontent.com/juanjodoblasm/proyecto_algoritmos_optimizacion/main/SEMINARIO/data_doblaje.csv'
df = pd.read_csv(url, header=1)

# 2. Limpieza de datos
# Quitamos la fila en blanco y la fila TOTAL y transformamos la columna a int
df['Toma'] = pd.to_numeric(df['Toma'], errors='coerce')
df = df.dropna(subset=['Toma'])
df['Toma'] = df['Toma'].astype(int)
# Guardamos la columna Toma como índice
df = df.set_index('Toma')
# Nos quedamos solo con las columnas de actores
df = df[[str(i) for i in range(1, 11)]]
# Columnas como enteros 1..10, con nombre "Actor"
df.columns = df.columns.astype(int)
df.columns.name = 'Actor'
# Los datos del dataframe los guardamos como enteros
df = df.astype(int)

# 3. Creación de conjuntos para cada toma con los actores que deben intervenir en cada una
tomas = {toma: set(fila.index[fila == 1]) for toma, fila in df.iterrows()}

Por último, como calcular una solución por fuerza bruta es insostenible (hay que iterar sobre las 726 mil trillones de particiones y aunque se procesaran mil millones de particiones por segundo, se tardaría del orden de 23 millones de años), mostraremos un ejemplo con las primeras 10 tomas.

In [44]:
# 4. Calculamos el coste de todas las particiones y mostramos aquellas con el coste mínimo
# (Ejemplo con las primeras 10 tomas)

# Lista donde guardaremos las particiones óptimas
argmins = []
# Variable donde guardaremos el coste mínimo
minimo = math.inf

# Generamos y recorremos las particiones y calculamos el mínimo y arg mínimos
for particion in generador_particiones(set(list(tomas)[:10]), 6):
    valor = f_objetivo(particion, tomas)

    if valor < minimo:
        argmins = [particion]
        minimo = valor

    elif valor == minimo:
        argmins.append(particion)

# Mostramos el resultado
print(f"Coste mínimo: {minimo}.")
print(f"Planificaciones que nos dan este coste mínimo:")
for plan in argmins:
    print(plan)

Coste mínimo: 13.
Planificaciones que nos dan este coste mínimo:
[{1, 2, 6, 7}, {3, 4, 5, 8, 9, 10}]
[{1, 2, 9, 6}, {3, 4, 5, 7, 8, 10}]
[{1, 2, 9, 7}, {3, 4, 5, 6, 8, 10}]
[{8, 1, 2, 10}, {3, 4, 5, 6, 7, 9}]
[{1, 2, 3, 4, 5}, {6, 7, 8, 9, 10}]
[{1, 2, 3, 6, 7}, {4, 5, 8, 9, 10}]
[{1, 2, 6, 7, 9}, {3, 4, 5, 8, 10}]
[{1, 2, 6, 8, 10}, {3, 4, 5, 7, 9}]
[{1, 2, 7, 8, 10}, {3, 4, 5, 6, 9}]
[{1, 2, 8, 9, 10}, {3, 4, 5, 6, 7}]
[{1, 2, 3, 4, 5, 6}, {8, 9, 10, 7}]
[{1, 2, 3, 4, 5, 7}, {8, 9, 10, 6}]
[{1, 2, 3, 4, 5, 9}, {8, 10, 6, 7}]
[{1, 2, 3, 4, 6, 7}, {8, 9, 10, 5}]
[{1, 2, 3, 6, 7, 9}, {8, 10, 4, 5}]
[{1, 2, 5, 6, 7, 9}, {8, 10, 3, 4}]
[{1, 2, 6, 7, 8, 10}, {9, 3, 4, 5}]
[{1, 2, 6, 8, 9, 10}, {3, 4, 5, 7}]
[{1, 2, 7, 8, 9, 10}, {3, 4, 5, 6}]


Calcula la complejidad del algoritmo por fuerza bruta

Respuesta

Sean $n$ el número de tomas, $m$ el número máximo de tomas que podemos grabar en un día y $p$ el número de actores. Una vez tenemos los datos limpios (un dataframe de dimensión $n \times p$), primero construimos el diccionario de tomas, recorriendo las $n$ filas del dataframe y añadiendo hasta $p$ actores al conjunto valor del diccionario, por lo que es del orden de $n \cdot p$.

Luego, tenemos un bucle donde recorremos las $B_n^{(m)}$ particiones del conjunto de tomas y en cada iteración llamamos a la función objetivo. Esta, con los dos primeros bucles está recorriendo las $n$ tomas y, dentro del segundo bucle, construimos el conjunto de actores, que puede tener hasta $p$ elementos. Por lo que la complejidad de la función objetivo tiene orden $O(n \cdot p)$.

De esta forma, la complejidad del algoritmo por fuerza bruta es del orden de $n \cdot p + B_n^{(m)} \cdot n \cdot p$, es decir, es $O(B_n^{(m)} \cdot n \cdot p)$.

(*)Diseña un algoritmo que mejore la complejidad del algortimo por fuerza bruta. Argumenta porque crees que mejora el algoritmo por fuerza bruta

Respuesta

Proponemos un algoritmo voraz para mejorar la complejidad frente a la fuerza bruta. Nuestra primera idea (`planificacion_voraz`) fue construir directamente una planificación de 5 días con 6 tomas cada uno, partiendo de que, como el coste depende del número de actores que se desplazan cada día y no del número de tomas grabadas, agrupar el máximo posible de tomas por día reduce el número de desplazamientos necesarios. Esto recorta drásticamente el espacio de búsqueda: en vez de recorrer todas las particiones posibles ($B_n^{(m)}$), solo consideramos las que tienen el menor número de grupos posible (en este caso, 5).

La estrategia de este primer algoritmo es la siguiente: se elige como primera toma del día la que requiere más actores (ya que, independientemente del día en que se grabe, esa toma va a "pagar" ese coste de todos modos, así que conviene asumirlo cuanto antes). A partir de ahí, se eligen las siguientes tomas del día minimizando en cada paso el número de actores nuevos que se añaden, para que el coste del día crezca lo menos posible.


In [45]:
def planificacion_voraz(tomas):
    """
    Genera una planificación de grabaciones agrupando siempre las tomas
    en días de tamaño máximo (6), eligiendo en cada paso la toma que
    añade menos actores nuevos al día en curso.

    input:
        tomas (dict): Diccionario {id_toma: set(id_actores)}.

    output:
        list: Lista de conjuntos (sets), donde cada conjunto contiene las tomas
              asignadas a un día de grabación.
    """

    # Preparamos el conjunto de tomas e inicializamos el plan de tomas diarias
    tomas_pendientes = set(tomas.keys())
    planificacion = []
  
    while tomas_pendientes:
        dia = set()
        actores_dia = set()
    
        # Se debe llenar el día actual con un máx de 6 tomas
        while len(dia) < 6 and tomas_pendientes:
            mejor_toma = None
            menor_incremento = float('inf')
      
            for t in tomas_pendientes:
                if not dia:
                    # Regla para que la primera toma del día sea la que más actores tenga
                    incremento = -len(tomas[t])
                else:
                    # Regla para el resto de tomas y conseguir los que añadan menos actores nuevos
                    nuevos_actores = tomas[t] - actores_dia
                    incremento = len(nuevos_actores)
        
                if incremento < menor_incremento:
                    menor_incremento = incremento
                    mejor_toma = t
      
            # Añadimos al conjunto de tomas de hoy la mejor toma detectada
            dia.add(mejor_toma)
            # Se añaden los actores extra
            actores_dia |= tomas[mejor_toma]
            # Se elimina del conjunto de tomas pendientes por grabar la toma que ha sido seleccionada
            tomas_pendientes.remove(mejor_toma)
    
        planificacion.append(dia)
  
    return planificacion

Sin embargo, la primera idea (`planificacion_voraz`) parte de una suposición que no está garantizada: que la solución óptima siempre agrupa las tomas en el menor número de días posible (5 días de 6 tomas). Esto no tiene por qué ser cierto — por ejemplo, un reparto en 6 días de 5 tomas cada uno también es válido, y según cómo se distribuyan los actores entre las tomas, podría tener un coste menor que cualquier reparto de 5×6.

Lo único que sí podemos afirmar con seguridad es que, dados dos grupos cualesquiera, fusionarlos (si el tamaño combinado no supera las 6 tomas) nunca empeora el coste: $|A_{G_1} \cup A_{G_2}| \le |A_{G_1}| + |A_{G_2}|$. Es decir, no sabemos de antemano cuál es la mejor cantidad de días, pero sí sabemos que fusionar grupos, cuando es posible, nunca perjudica.

Con esta idea diseñamos un segundo algoritmo (`planificacion_voraz2`), en dos fases:

1. **Fase 1:** en vez de forzar grupos de 6 tomas, construimos cada grupo empezando por la toma con más actores y añadiendo, mientras sea posible, tomas cuyo conjunto de actores ya esté completamente incluido en los actores del grupo (es decir, tomas que no añaden ningún coste extra), priorizando en cada paso la toma "gratis" con más actores. Un grupo se cierra en cuanto no queda ninguna toma "gratis" que añadir, aunque tenga menos de 6 tomas.

2. **Fase 2:** una vez formados todos los grupos, intentamos fusionarlos entre sí (mientras el tamaño combinado no supere las 6 tomas), priorizando la fusión que produzca el menor coste resultante, y en caso de empate, la que junte más tomas. Repetimos esto hasta que no quede ninguna fusión posible.


In [46]:
def planificacion_voraz2(tomas):
    """
    Genera una planificación de grabaciones en dos fases: primero agrupa
    tomas sin coste adicional (fase 1) y después fusiona los grupos
    resultantes siempre que sea posible sin superar el máximo de tomas
    por día (fase 2).

    input:
        tomas (dict): Diccionario {id_toma: set(id_actores)}.

    output:
        list: Lista de conjuntos (sets), donde cada conjunto contiene las tomas
              asignadas a un día de grabación.
    """
    # Preparamos el conjunto de tomas e inicializamos el plan de tomas diarias
    tomas_pendientes = set(tomas.keys())
    planificacion = []
  
    # Fase 1: agrupar tomas sin coste adicional
    while tomas_pendientes:
        # Elegimos como ancla del día la toma pendiente con más actores
        toma_mas_actores = max(tomas_pendientes, key=lambda x : len(tomas[x]))
        tomas_pendientes.remove(toma_mas_actores)
        dia = {toma_mas_actores}
        # Copiamos su conjunto de actores (nunca se muta, no hace falta actualizarlo)
        actores_dia = tomas[toma_mas_actores].copy()
    
        # Añadimos tomas cuyo conjunto de actores ya está cubierto por actores_dia
        # (coste marginal cero), priorizando las que más actores tienen
        while len(dia) < 6 and tomas_pendientes:
            mejor_toma = max(
                {t for t in tomas_pendientes if tomas[t].issubset(actores_dia)},
                key=lambda x : len(tomas[x]),
                default=None
            )
            
            if mejor_toma is None:
                break
      
            dia.add(mejor_toma)
            tomas_pendientes.remove(mejor_toma)
    
        planificacion.append(dia)
    
    # Fase 2: fusionar días mientras el tamaño combinado no supere 6 tomas
    i = 0
    while i < len(planificacion) - 1:
        dia1 = planificacion[i]
        j = i + 1
        mejor_dia2 = planificacion[j]
        menor_coste = np.inf

        # Buscamos, entre los días posteriores, el mejor candidato para fusionar
        while j < len(planificacion):
            dia2 = planificacion[j]
            union = dia1 | dia2

            if len(union) <= 6:
                coste = f_objetivo([union], tomas)

                # Nos quedamos con la fusión de menor coste resultante
                if coste < menor_coste:
                    menor_coste = coste
                    mejor_dia2 = dia2

                # En caso de empate, preferimos la que junte más tomas
                elif coste == menor_coste and len(dia2) > len(mejor_dia2):
                    mejor_dia2 = dia2

            j += 1

        # Aplicamos la fusión y NO avanzamos i, para reintentar fusionar
        # el día ya crecido con lo que quede    
        if menor_coste < np.inf:
            dia1 |= mejor_dia2
            planificacion.remove(mejor_dia2)
    
        # Ninguna fusión posible para dia1: pasamos al siguiente día
        else:
            i += 1
  
    return planificacion

Vamos a probar los dos algoritmos en dos pasos. Primero, los aplicamos al mismo subconjunto de 10 tomas que usamos para la fuerza bruta, para comprobar si la planificación que devuelven coincide con alguna de las soluciones óptimas que ya guardamos y por lo tanto alcanzan el coste óptimo real (13). Después, aplicamos ambos algoritmos al conjunto completo de 30 tomas y comparamos sus resultados entre sí.

Al ser ambos heurísticos, ninguno de los dos está garantizado a ser mejor que el otro en general: cuál da mejor resultado depende de los datos concretos, así que la comparación que sigue solo es válida para esta instancia del problema, no como conclusión universal.

In [47]:
tomas_10 = {toma: tomas[toma] for toma in list(tomas)[:10]}

plan_voraz_10 = planificacion_voraz(tomas_10)
coste_voraz_10 = f_objetivo(plan_voraz_10, tomas_10)

plan_voraz2_10 = planificacion_voraz2(tomas_10)
coste_voraz2_10 = f_objetivo(plan_voraz2_10, tomas_10)

print(f"Coste óptimo (fuerza bruta): {minimo}")
print(f"`planificacion_voraz`  -> coste: {coste_voraz_10}, óptimo: {coste_voraz_10 == minimo}, "
      f"coincide con alguna solución óptima: {plan_voraz_10 in argmins}")
print(f"Solución encontrada: {plan_voraz_10}")
print(f"`planificacion_voraz2` -> coste: {coste_voraz2_10}, óptimo: {coste_voraz2_10 == minimo}, "
      f"coincide con alguna solución óptima: {plan_voraz2_10 in argmins}")
print(f"Solución encontrada: {plan_voraz2_10}")


Coste óptimo (fuerza bruta): 13
`planificacion_voraz`  -> coste: 13, óptimo: True, coincide con alguna solución óptima: True
Solución encontrada: [{1, 2, 3, 6, 7, 9}, {8, 10, 4, 5}]
`planificacion_voraz2` -> coste: 13, óptimo: True, coincide con alguna solución óptima: True
Solución encontrada: [{1, 2, 3, 6, 7, 9}, {8, 10, 4, 5}]


Como ambos algoritmos alcanzan el coste óptimo en el ejemplo con las 10 primeras tomas, los aplicamos ahora al problema completo (30 tomas), donde ya no podemos comparar contra fuerza bruta por ser inviable, pero sí podemos comparar los dos algoritmos entre sí.

In [ ]:
planificacion = planificacion_voraz(tomas)
print(f'La planificación de `planificacion_voraz` es la siguiente:')
for i, dia in enumerate(planificacion):
    print(f'Día {i+1}: {dia}')
coste_voraz = f_objetivo(planificacion, tomas)
print(f'\nEl coste de la planificación calculada es:\n{coste_voraz}')

planificacion2 = planificacion_voraz2(tomas)
print(f'\nLa planificación de `planificacion_voraz2` es la siguiente:')
for i, dia in enumerate(planificacion2):
    print(f'Día {i+1}: {dia}')
coste_voraz2 = f_objetivo(planificacion2, tomas)
print(f'\nEl coste de la planificación calculada es:\n{coste_voraz2}')

La mejor planificación calculada es la siguiente:
Día 1: {1, 2, 6, 7, 9, 13}
Día 2: {3, 4, 11, 17, 19, 23}
Día 3: {8, 12, 14, 18, 22, 24}
Día 4: {5, 10, 15, 21, 28, 30}
Día 5: {16, 20, 25, 26, 27, 29}

El coste de la planificación calculada es:
31

La planificación de planificacion_voraz2 es la siguiente:
Día 1: {1, 2, 6, 7, 20, 22}
Día 2: {17, 19, 4, 23, 11, 15}
Día 3: {8, 9, 12, 14, 18, 24}
Día 4: {29, 26, 3, 10, 27, 13}
Día 5: {16, 5, 21, 25, 28, 30}

El coste de la planificación calculada es:
30


Sorprendentemente, `planificacion_voraz2` obtiene un coste menor que `planificacion_voraz`, a pesar de no forzar de entrada la estructura de 5 días de 6 tomas — de hecho, tras la fase de fusión, termina convergiendo igualmente a esa misma estructura (5×6), aunque llegando a ella por un camino distinto (agrupando primero por coste cero y fusionando después, en vez de fijar el tamaño de grupo desde el principio). Esto no contradice lo que planteamos antes: 5×6 no está garantizado como óptimo en general, pero en este conjunto de datos concreto resulta ser la mejor estructura que encuentran ambos enfoques.

Para intentar mejorar aún más estos resultados, aplicamos una búsqueda local: partiendo de una planificación ya construida, exploramos pequeñas modificaciones —en este caso, intercambiar una toma entre dos días— y nos quedamos con cualquier intercambio que reduzca el coste. Repetimos este proceso hasta que ningún intercambio posible consiga mejorarlo más, momento en el que decimos que hemos alcanzado un óptimo local (no necesariamente el óptimo global del problema, pero sí el mejor resultado alcanzable moviéndose paso a paso desde donde se empezó).

In [49]:
def busqueda_local(planificacion_inicial, tomas):
    """
    Optimiza una planificación inicial mediante un algoritmo de Búsqueda Local (Swap).
    Utiliza la estrategia del "Primer Mejor" (First Improvement) para escapar
    de subóptimos evaluando el intercambio de pares de tomas entre distintos días.

    input:
        planificacion_inicial (list): Lista de conjuntos con la distribución inicial de tomas por día.
        tomas (dict): Diccionario {id_toma: set(id_actores)}.

    output:
        tuple: (mejor_planificacion, mejor_coste)
            - mejor_planificacion (list): La planificación optimizada (óptimo local).
            - mejor_coste (int): El coste final asociado a dicha planificación.
    """

    # Copia de la planificación calculada anteriormente y no alterar la original
    # en caso de no encontrar una solución mejor
    mejor_planificacion = copy.deepcopy(planificacion_inicial)
    # Inicializamos el mejor coste con el coste obtenido anteriormente
    mejor_coste = f_objetivo(mejor_planificacion, tomas)
    # Flag que permite hacer la búsqueda local
    mejora = True
  
    iteracion = 1
  
    print(f"\nInicio de búsqueda local")
    print(f"Coste inicial a mejorar: {mejor_coste}")
  
    while mejora:
        # Bajamos la flag. En caso de encontrar una mejora no será necesaria continuar
        mejora = False
        # Exploramos todas las combinaciones de días i y j
        for i in range(len(mejor_planificacion)):
            for j in range(i + 1, len(mejor_planificacion)):
                dia_i = mejor_planificacion[i]
                dia_j = mejor_planificacion[j]
        
                # Probamos a intercambiar tomas t_i y t_j
                for t_i in dia_i:
                    for t_j in dia_j:
                        # Creamos una copia temporal de la planificación para evaluar el movimiento
                        plan_temp = copy.deepcopy(mejor_planificacion)
                        # Hacemos el intercambio
                        plan_temp[i].remove(t_i)
                        plan_temp[i].add(t_j)
                        plan_temp[j].remove(t_j)
                        plan_temp[j].add(t_i)
            
                        coste_temp = f_objetivo(plan_temp, tomas)
            
                        # Si se encuentra un coste mejor, se da por bueno
                        if coste_temp < mejor_coste:
                            print(f"Iteración {iteracion}: ¡Mejora! Coste baja de {mejor_coste} a {coste_temp} (Intercambio: Toma {t_i} por Toma {t_j})")
                            mejor_planificacion = plan_temp
                            mejor_coste = coste_temp
                            # Reiniciamos la búsqueda con la nueva solución
                            mejora = True
                            iteracion += 1
                            break
          
                    if mejora: break
                if mejora: break
            if mejora: break
    
    if iteracion == 1:
        print("\nBúsqueda finalizada")
        print("La búsqueda local exploró todas las combinaciones y no encontró más mejoras.")
    else:
        print("\nBúsqueda finalizada")
        print(f"Coste final optimizado: {mejor_coste}")
  
    return mejor_planificacion, mejor_coste

A continuación, aplicamos esta búsqueda local tanto sobre el resultado de `planificacion_voraz` como sobre el de `planificacion_voraz2`, para comprobar si alguno de los dos se puede seguir mejorando.

In [51]:
print('Planificación inicial a evaluar en la búsqueda local:')
for i, dia in enumerate(planificacion):
    print(f'Día {i+1}: {dia}')

plan_final, coste_final = busqueda_local(planificacion, tomas)
for i, dia in enumerate(plan_final):
    print(f'Día {i+1}: {dia}')
print(f'El coste final es: {coste_final}')

Planificación inicial a evaluar en la búsqueda local:
Día 1: {1, 2, 6, 7, 9, 13}
Día 2: {3, 4, 11, 17, 19, 23}
Día 3: {8, 12, 14, 18, 22, 24}
Día 4: {5, 10, 15, 21, 28, 30}
Día 5: {16, 20, 25, 26, 27, 29}

Inicio de búsqueda local
Coste inicial a mejorar: 31

Búsqueda finalizada
La búsqueda local exploró todas las combinaciones y no encontró más mejoras.
Día 1: {1, 2, 6, 7, 9, 13}
Día 2: {3, 4, 11, 17, 19, 23}
Día 3: {8, 12, 14, 18, 22, 24}
Día 4: {5, 10, 15, 21, 28, 30}
Día 5: {16, 20, 25, 26, 27, 29}
El coste final es: 31


In [53]:
print('\nPlanificación inicial (voraz2) a evaluar en la búsqueda local:')
for i, dia in enumerate(planificacion2):
    print(f'Día {i+1}: {dia}')

plan2_final, coste2_final = busqueda_local(planificacion2, tomas)
for i, dia in enumerate(plan2_final):
    print(f'Día {i+1}: {dia}')
print(f'El coste final es: {coste2_final}')


Planificación inicial (voraz2) a evaluar en la búsqueda local:
Día 1: {1, 2, 6, 7, 20, 22}
Día 2: {17, 19, 4, 23, 11, 15}
Día 3: {8, 9, 12, 14, 18, 24}
Día 4: {29, 26, 3, 10, 27, 13}
Día 5: {16, 5, 21, 25, 28, 30}

Inicio de búsqueda local
Coste inicial a mejorar: 30
Iteración 1: ¡Mejora! Coste baja de 30 a 29 (Intercambio: Toma 17 por Toma 3)

Búsqueda finalizada
Coste final optimizado: 29
Día 1: {1, 2, 6, 7, 20, 22}
Día 2: {3, 4, 11, 15, 19, 23}
Día 3: {8, 9, 12, 14, 18, 24}
Día 4: {10, 13, 17, 26, 27, 29}
Día 5: {5, 16, 21, 25, 28, 30}
El coste final es: 29


In [54]:
print(f'\nResumen comparativo (30 tomas):')
print(f'planificacion_voraz  + búsqueda local -> coste: {coste_final}')
print(f'planificacion_voraz2 + búsqueda local -> coste: {coste2_final}')


Resumen comparativo (30 tomas):
planificacion_voraz  + búsqueda local -> coste: 31
planificacion_voraz2 + búsqueda local -> coste: 29


Podemos ver que `busqueda_local` no encuentra ninguna mejora sobre la planificación de `planificacion_voraz` (se queda en 31), lo que indica que esa solución ya era un óptimo local respecto a intercambios de una toma entre días. En cambio, sí consigue mejorar la planificación de `planificacion_voraz2`, bajando su coste de 30 a 29 con un único intercambio — es decir, ese punto de partida no era todavía un óptimo local, y ahí la búsqueda local sí aporta valor real.

En conjunto, la combinación `planificacion_voraz2` + búsqueda local (coste 29) es la mejor solución que hemos encontrado entre las cuatro combinaciones probadas, reforzando la conclusión de que no asumir de entrada la estructura de 5 días de 6 tomas permite explorar mejores soluciones.

(*)Calcula la complejidad del algoritmo

Respuesta

Al igual que antes, sean $n$ el número de tomas, $m$ el número máximo de tomas por día y $p$ el número de actores. Analizamos por separado la complejidad de cada uno de los tres algoritmos, y después las combinaciones que hemos probado.

**`planificacion_voraz`:** el algoritmo forma $\lceil n/m \rceil = O(n)$ días (tratando $m$ como constante). Por cada una de las $O(n)$ tomas seleccionadas en total, se recorre el conjunto de tomas pendientes ($O(n)$) calculando para cada una una diferencia de conjuntos de coste $O(p)$. En total: $O(n^2 \cdot p)$.

**`planificacion_voraz2`:**
- *Fase 1:* el bucle exterior forma como mucho $O(n)$ grupos (en el caso extremo de que cada grupo acabe teniendo solo la toma ancla). Dentro de cada iteración, elegir la toma ancla cuesta $O(n)$, pero ese coste queda absorbido por el del bucle interior: este se repite como mucho $m=6$ veces (constante) y, en cada repetición, recorre las tomas pendientes ($O(n)$) comprobando para cada una si es subconjunto de los actores del día (coste $O(p)$), es decir $O(n\cdot p)$ por grupo. Sumando las $O(n)$ iteraciones del bucle exterior: $O(n) \times O(n\cdot p) = O(n^2\cdot p)$.
- *Fase 2:* en el peor caso hay $O(n)$ grupos a fusionar, y por cada intento de fusión (a lo sumo $O(n)$ en total) se recorren los demás grupos ($O(n)$), calculando el coste de cada posible unión ($O(m\cdot p) = O(p)$, con $m$ constante). En total: $O(n) \times O(n) \times O(p) = O(n^2 \cdot p)$.
- Sumando ambas fases: $O(n^2 \cdot p)$ — el mismo orden que `planificacion_voraz`.

**`busqueda_local`:** por cada barrido completo, se comparan todos los pares de días ($O((n/m)^2)$) y, por cada par, todos los pares de tomas entre ambos días ($O(m^2)$); estos dos factores se cancelan, dejando $O(n^2)$ intercambios evaluados por barrido, cada uno con coste $O(n\cdot p)$ (recalcular la función objetivo sobre toda la planificación). Como cada intercambio de mejora reduce el coste en al menos una unidad y el coste máximo posible está acotado por $O(n\cdot p)$, el número de barridos está acotado por $O(n\cdot p)$. En total: $O(n^2) \times O(n\cdot p) \times O(n\cdot p) = O(n^4 \cdot p^2)$.

**Complejidad combinada:** como cada algoritmo voraz se ejecuta una vez y después se le aplica la búsqueda local de forma secuencial, tanto en `planificacion_voraz` + búsqueda local como en `planificacion_voraz2` + búsqueda local la complejidad total es $O(n^2\cdot p) + O(n^4\cdot p^2) = O(n^4 \cdot p^2)$, dominada en los dos casos por la fase de búsqueda local.

Esto plantea una cuestión de coste-beneficio: la búsqueda local es varios órdenes de magnitud más costosa que cualquiera de los dos algoritmos voraces, y sin embargo la mejora que consigue en la práctica es modesta (nula sobre `planificacion_voraz`, y de una sola unidad —de 30 a 29— sobre `planificacion_voraz2`). Para un problema de este tamaño, cabe preguntarse si el coste computacional añadido compensa esa mejora, y en instancias más grandes esta desproporción se acentuaría todavía más.


Según el problema (y tenga sentido), diseña un juego de datos de entrada aleatorios

Respuesta

Para comprobar que los algoritmos generalizan más allá del caso concreto del enunciado, generamos un juego de datos aleatorio con las mismas dimensiones (30 tomas, 10 actores).

In [56]:
def generador_datos_rodaje(num_tomas, num_actores):
    """
    Genera un conjunto de datos sintético y aleatorio que simula la participación
    de actores en diferentes tomas de un rodaje.

    input:
        num_tomas (int): Cantidad total de tomas que se van a grabar.
        num_actores (int): Cantidad total de actores disponibles para el rodaje.

    output:
        pandas.DataFrame: Matriz binaria donde:
            - El índice ('Toma') representa el número secuencial de la toma.
            - Las columnas (etiquetadas de '1' a 'num_actores') representan a cada actor.
            - El valor es 1 si el actor participa en la toma, y 0 en caso contrario.
    """
    # Fijamos la semilla para que el resultado sea reproducible en la entrega
    np.random.seed(0)

    # 1. Generamos la secuencia de tomas (del 1 al valor de num_tomas)
    columna_tomas = np.arange(1, num_tomas + 1)

    # 2. Generamos una matriz aleatoria de ceros y unos para las columnas de los actores
    # np.random.randint(0, 2) genera números aleatorios incluyendo el 0 y excluyendo el 2
    matriz_aleatoria = np.random.randint(0, 2, size=(num_tomas, num_actores))

    # 3. Creamos el DataFrame estructurado con los nombres de columna secuenciales
    nombres_columnas_actores = [str(i) for i in range(1, num_actores + 1)]
    df_aleatorio = pd.DataFrame(matriz_aleatoria, columns=nombres_columnas_actores)

    # 4. Insertamos la columna "Toma", la rellenamos y la configuramos como índice
    df_aleatorio.insert(0, 'Toma', columna_tomas)
    df_aleatorio.set_index('Toma', inplace=True)

    return df_aleatorio

In [57]:
# Generamos un juego de datos aleatorios de las mismas dimensiones
# aportadas en el enunciado: 30 tomas y 10 actores involucrados.
datos_aleatorios = generador_datos_rodaje(30, 10)

# Creación de conjuntos para cada toma con los actores que deben intervenir en cada una
tomas_datos_aleatorios = {toma: set(fila.index[fila == 1]) for toma, fila in datos_aleatorios.iterrows()}

Aplica el algoritmo al juego de datos generado

Respuesta

Aplicamos sobre el juego de datos aleatorios los dos algoritmos voraces junto con la búsqueda local, igual que hicimos con los datos reales.

In [59]:
# Aplicamos primero planificacion_voraz para obtener una solución inicial
# sobre el conjunto de datos aleatorios generados
plani_prueba = planificacion_voraz(tomas_datos_aleatorios)
coste_inicial = f_objetivo(plani_prueba, tomas_datos_aleatorios)
print('planificacion_voraz:')
for i, dia in enumerate(plani_prueba):
    print(f'Día {i+1}: {dia}')
print(f'Coste inicial: {coste_inicial}')

plani_prueba_local, coste_prueba = busqueda_local(plani_prueba, tomas_datos_aleatorios)
print(f'\nCoste tras búsqueda local: {coste_prueba}')

planificacion_voraz:
Día 1: {2, 6, 7, 10, 11, 13}
Día 2: {1, 3, 4, 8, 14, 17}
Día 3: {5, 9, 12, 16, 23, 25}
Día 4: {15, 18, 19, 20, 22, 28}
Día 5: {21, 24, 26, 27, 29, 30}
Coste inicial: 46

Inicio de búsqueda local
Coste inicial a mejorar: 46
Iteración 1: ¡Mejora! Coste baja de 46 a 45 (Intercambio: Toma 7 por Toma 20)
Iteración 2: ¡Mejora! Coste baja de 45 a 44 (Intercambio: Toma 2 por Toma 26)
Iteración 3: ¡Mejora! Coste baja de 44 a 43 (Intercambio: Toma 10 por Toma 1)
Iteración 4: ¡Mejora! Coste baja de 43 a 42 (Intercambio: Toma 8 por Toma 21)

Búsqueda finalizada
Coste final optimizado: 42

Coste tras búsqueda local: 42


In [61]:
# Repetimos con planificacion_voraz2
plani_prueba2 = planificacion_voraz2(tomas_datos_aleatorios)
coste_inicial2 = f_objetivo(plani_prueba2, tomas_datos_aleatorios)
print('\nplanificacion_voraz2:')
for i, dia in enumerate(plani_prueba2):
    print(f'Día {i+1}: {dia}')
print(f'Coste inicial: {coste_inicial2}')

plani_prueba2_local, coste_prueba2 = busqueda_local(plani_prueba2, tomas_datos_aleatorios)
print(f'\nCoste tras búsqueda local: {coste_prueba2}')


planificacion_voraz2:
Día 1: {6, 10, 13, 15, 28, 30}
Día 2: {1, 17, 3, 23, 9, 14}
Día 3: {16, 2, 4, 5, 22, 25}
Día 4: {7, 18, 19, 21, 26}
Día 5: {8, 12, 24, 27, 29}
Día 6: {11, 20}
Coste inicial: 45

Inicio de búsqueda local
Coste inicial a mejorar: 45
Iteración 1: ¡Mejora! Coste baja de 45 a 44 (Intercambio: Toma 4 por Toma 26)

Búsqueda finalizada
Coste final optimizado: 44

Coste tras búsqueda local: 44


In [62]:
print(f'\nResumen comparativo (datos aleatorios):')
print(f'planificacion_voraz  + búsqueda local -> coste: {coste_prueba}')
print(f'planificacion_voraz2 + búsqueda local -> coste: {coste_prueba2}')


Resumen comparativo (datos aleatorios):
planificacion_voraz  + búsqueda local -> coste: 42
planificacion_voraz2 + búsqueda local -> coste: 44


En los datos aleatorios, `planificacion_voraz2` no converge a la estructura de 5 días de 6 tomas — en la fase de fusión deja seis grupos en vez de cinco (tamaños 6, 6, 6, 5, 5 y 2), reflejando que aquí esa estructura tampoco se puede alcanzar usando solo fusiones sin coste añadido. Y, a diferencia de los datos reales, en este caso es `planificacion_voraz` con búsqueda local (coste 42) quien supera a `planificacion_voraz2` con búsqueda local (coste 44).

Este resultado es la confirmación empírica de lo que planteamos antes: ningún algoritmo voraz domina al otro en general, y cuál funciona mejor depende de cómo estén distribuidos los actores entre las tomas en cada instancia concreta. Con los datos reales ganaba `planificacion_voraz2`; con estos datos aleatorios gana `planificacion_voraz`.

Enumera las referencias que has utilizado(si ha sido necesario) para llevar a cabo el trabajo

Respuesta

1. **Documentación oficial de Python:** para el manejo profundo de bibliotecas integradas utilizadas en el código, como [itertools](https://docs.python.org/es/3.14/library/itertools.html) (específicamente combinations) y [functools](https://docs.python.org/es/3.14/library/functools.html#module-functools) (para lru_cache).

2. **Combinatoria y Matemática Discreta:** conceptos sobre particiones de conjuntos y la sucesión de los [Números de Bell](https://es.wikipedia.org/wiki/N%C3%BAmero_de_Bell) ($B_n$), que sustentan las fórmulas recursivas utilizadas para calcular la magnitud exacta del espacio de soluciones.

3. **Asistencia mediante IA:** durante el desarrollo del proyecto se han empleado modelos de IA generativa como asistente de programación. Su uso se ha enfocado específicamente en el apoyo para la depuración de código, la optimización de la sintaxis en Python y la exploración iterativa de alternativas para el diseño de las heurísticas. Todas las sugerencias y fragmentos generados han sido revisados, validados y adaptados de forma crítica para garantizar su correcta integración en la resolución del problema planteado.

Describe brevemente las lineas de como crees que es posible avanzar en el estudio del problema. Ten en cuenta incluso posibles variaciones del problema y/o variaciones al alza del tamaño

Respuesta

En cuanto a posibles variaciones del problema y posibles formas de abordaje, hemos identificado 4 componentes para tener en cuenta a futuro:

1. **Escalabilidad y aplicación de Metaheurística:** el problema tiene una complejidad computacional dominada por $O(n^4 \cdot p^2)$ si se aplica búsqueda local, o por $O(n^2 \cdot p)$ si solo se usa el algoritmo voraz. Si se produce una variación al alza del tamaño (por ejemplo, pasar de 30 tomas y 10 actores a producciones masivas con cientos de tomas y decenas de actores), la búsqueda local exhaustiva será ineficiente. El siguiente paso lógico sería implementar metaheurísticas avanzadas como Algoritmos Genéticos o Recocido Simulado, que permiten explorar espacios de búsqueda inmensos y escapar de óptimos locales en tiempos de ejecución razonables.

2. **Costes Heterogéneos de los Actores:** el modelo actual simplifica el cálculo asumiendo que todos los actores de doblaje cobran la misma cantidad por desplazamiento al estudio. Una variación altamente realista consistiría en asignar un "caché" o tarifa diaria diferente a cada actor. Esto cambiaría drásticamente la forma en la que los algoritmos voraces seleccionan las tomas, ya que priorizarían agrupar a los actores más caros en la menor cantidad de días posibles.

3. **Limitaciones por Capacidad Horaria:** actualmente, la limitación del problema establece que no es posible grabar más de 6 tomas por día. Un avance sería dotar a cada toma de una duración estimada específica (en minutos u horas) y sustituir la restricción de "tomas por día" por una capacidad máxima de horas de rodaje por jornada.

4. **Relaciones de Precedencia:** añadir dependencias entre tomas. Por razones narrativas, de preparación del set o de continuidad, algunas tomas podrían requerir grabarse estrictamente antes o después de otras, limitando las combinaciones válidas en la creación de particiones iniciales.